# CS Resource Hub EDA

This notebook explores the generated CS Resource Hub dataset. It is optional and does not replace the validation scripts; use it to inspect coverage, metadata quality, duplicate candidates, and domain concentration before expanding or presenting the dataset.

## Setup

Run this notebook from the repository root after regenerating exports with the normal project commands. The notebook reads `generated/all_resources.json`, `generated/resources.csv`, and `generated/stats.json`.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import sys

import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "generated" / "all_resources.json").exists():
    ROOT = ROOT.parent

scripts_path = ROOT / "scripts"
if str(scripts_path) not in sys.path:
    sys.path.insert(0, str(scripts_path))

from utils import CATEGORY_GROUPS

all_resources_path = ROOT / "generated" / "all_resources.json"
stats_path = ROOT / "generated" / "stats.json"

with all_resources_path.open(encoding="utf-8") as f:
    payload = json.load(f)

with stats_path.open(encoding="utf-8") as f:
    stats = json.load(f)

df = pd.DataFrame(payload["resources"])

def domain_from_url(url: object) -> str:
    return urlparse(str(url)).netloc.removeprefix("www.")

df["domain"] = df["url"].map(domain_from_url)
df["description_length"] = df["description"].str.len()
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df["last_verified"] = pd.to_datetime(df["last_verified"], errors="coerce")
df["days_since_verified"] = (pd.Timestamp.today().normalize() - df["last_verified"]).dt.days

plt.style.use("default")
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)

print(f"Loaded {len(df):,} resources from {all_resources_path.relative_to(ROOT)}")

## 1. Dataset Overview

Start with the dataset size, generated date, available fields, and a small sample. This confirms the notebook is reading the current generated exports.

In [ ]:
overview = pd.DataFrame(
    {
        "metric": ["generated", "total_resources", "columns", "categories"],
        "value": [
            payload.get("generated"),
            len(df),
            len(df.columns),
            df["category"].nunique(),
        ],
    }
)
display(overview)
display(df.head(10))

## 2. Category & Group Coverage

These views show whether the dataset is balanced across the project categories and broader resource groups.

In [ ]:
category_to_group = {
    category: group
    for group, categories in CATEGORY_GROUPS.items()
    for category in categories
}

df["group"] = df["category"].map(category_to_group)

category_counts = df["category"].value_counts().sort_values()
group_counts = df["group"].value_counts().sort_values()

display(category_counts.sort_values(ascending=False).rename("resources"))
display(group_counts.sort_values(ascending=False).rename("resources"))

ax = category_counts.plot(kind="barh", figsize=(9, 6), title="Resources by Category")
ax.set_xlabel("Resources")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()

ax = group_counts.plot(kind="barh", figsize=(8, 3), title="Resources by Group")
ax.set_xlabel("Resources")
ax.set_ylabel("Group")
plt.tight_layout()
plt.show()

## 3. Type & Metadata Breakdown

This section checks which resource types dominate the dataset and how optional metadata such as `month` and `location` is used.

In [ ]:
type_counts = df["type"].fillna("missing").value_counts()
metadata_presence = df[["type", "month", "location"]].notna().sum().rename("present")
metadata_missing = df[["type", "month", "location"]].isna().sum().rename("missing")

display(type_counts.rename("resources"))
display(pd.concat([metadata_presence, metadata_missing], axis=1))

ax = type_counts.head(15).sort_values().plot(
    kind="barh",
    figsize=(9, 6),
    title="Top Resource Types",
)
ax.set_xlabel("Resources")
ax.set_ylabel("Type")
plt.tight_layout()
plt.show()

display(
    df.groupby("category")[["type", "month", "location"]]
    .apply(lambda frame: frame.notna().sum())
    .sort_index()
)

## 4. Data Quality Checks

These checks are exploratory. Validation still belongs in `scripts/`, but this view helps spot patterns that may deserve stricter checks later.

In [ ]:
required_fields = ["id", "name", "url", "description", "category", "date_added", "last_verified"]
missing_required = df[required_fields].isna().sum()

quality_summary = pd.DataFrame(
    {
        "check": [
            "missing_required_values",
            "duplicate_ids",
            "duplicate_urls",
            "descriptions_under_10_chars",
            "descriptions_over_120_chars",
            "unverified_over_180_days",
        ],
        "count": [
            int(missing_required.sum()),
            int(df["id"].duplicated().sum()),
            int(df["url"].duplicated().sum()),
            int((df["description_length"] < 10).sum()),
            int((df["description_length"] > 120).sum()),
            int((df["days_since_verified"] > 180).sum()),
        ],
    }
)

display(quality_summary)
display(missing_required.rename("missing_required_values"))

stale = df.loc[df["days_since_verified"] > 180, ["name", "category", "url", "last_verified", "days_since_verified"]]
display(stale.sort_values("days_since_verified", ascending=False).head(20))

## 5. Duplicate & Domain Analysis

This section looks for repeated names, repeated URLs, and domains that appear often. Repeated domains are not always bad, but high concentration is useful to know.

In [ ]:
duplicate_names = df[df["name"].str.lower().duplicated(keep=False)].sort_values("name")
duplicate_urls = df[df["url"].duplicated(keep=False)].sort_values("url")
domain_counts = df["domain"].value_counts()

display(duplicate_names[["name", "category", "url"]])
display(duplicate_urls[["name", "category", "url"]])
display(domain_counts.head(20).rename("resources"))

ax = domain_counts.head(15).sort_values().plot(
    kind="barh",
    figsize=(9, 5),
    title="Most Common Domains",
)
ax.set_xlabel("Resources")
ax.set_ylabel("Domain")
plt.tight_layout()
plt.show()

## Notes For Future Dataset Work

Use the outputs above to decide where to add resources next, whether any metadata should become stricter, and whether repeated domains or duplicate candidates need manual review. Keep durable validation rules in `scripts/` after a notebook finding proves useful.